In [ ]:
import os
import sys; sys.path.append(os.path.join(os.path.abspath(''), '../experiments'));

import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import parallelism, multiprocessing, itertools, setproctitle
import os, time, numpy as np

import json


In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False
stiffness_pressure = 0.4
scale_factor_pressure = 0.01
avg_len = 0.65

In [ ]:
tag = '0.10_1.5_60.00'

amp = float(tag.split('_')[0])
r = float(tag.split('_')[1])
angle = float(tag.split('_')[2])

In [ ]:
%%capture

h = 5
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])
ipu, m, marker = pattern_generator_using_gmsh.get_cosine_dash(h, avg_len, avg_len, amplitude=amp, dash_point = dash_point)

finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
configure_solver_parallelism()

In [ ]:
framerate = 5 # Update every 5 iterations
def cb(it):
    # if it % framerate == 0:
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
hessianShiftForRigidMotion = 1e-10
hessianShiftForAlphainPlanar = 1e-12

In [ ]:
fixedVars, hessianShift = [], hessianShiftForRigidMotion

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

In [ ]:
opts.niter = 500
opts.gradTol = 1e-10

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

In [ ]:
fixedVars, hessianShift = list(ipu.get_center_fixedVars()), hessianShiftForAlphainPlanar

In [ ]:
opts.niter = 500
opts.gradTol = 1e-10

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

In [ ]:
experiment_log = {}
experiment_log["Ipu simulation succeed"] = int(cr.success)
experiment_log["Simulation Kappa value"] = (ipu.getVars()[-2])
if np.abs(ipu.getVars()[-2]) > 1e-10:
    experiment_log["Planar equilibrium"] = 0
    print("Warning: Can not compute stiffness due to non-planar equilibrium!")
else:
    experiment_log["Planar equilibrium"] = 1

In [ ]:
result_folder = 'output'
name = 'cosine_dash'
variable = 0
render_images = True

In [ ]:
# # Compute stiffness after removing the vertical offset
# ipu.reparametrize_vertical_offset()
# optimizer = inflation.get_inflation_optimizer(ipu, ipu.getBendingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)
# stiffness_values, sampled_alphas, stiffness_coefficient = visualize_sampled_bending_stiffness(ipu, 1000, optimizer, hessianShift = 0, fixedVars = ipu.getBendingStiffnessFixedVars(), filename = "stiffness.png", generate_images = render_images)

In [ ]:
# from IPython.display import Image
# Image(filename="stiffness.png") 

In [ ]:
np.set_printoptions(suppress=True, precision=6)

In [ ]:
import periodic_simulation_setup, importlib
importlib.reload(periodic_simulation_setup)

In [ ]:
ipu.reparametrize_vertical_offset()


az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, useTFT, disableFusedRegionTFT)
az_optimizer = inflation.get_inflation_optimizer(az_ipu, az_ipu.getBendingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)

In [ ]:
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)
az_viewer.show()

In [ ]:
framerate = 5 # Update every 5 iterations
def az_cb(it):
    # if it % framerate == 0:
    az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
cr = inflation.inflation_newton(az_ipu, az_ipu.getRigidMotionFixedVars(), opts, callback=az_cb, hessianShift = hessianShift)

In [ ]:
import periodic_simulation_setup
import importlib
importlib.reload(periodic_simulation_setup)

In [ ]:
betas = np.linspace(0, 2 * np.pi, 1000)

In [ ]:
optimizer = inflation.get_inflation_optimizer(ipu, ipu.getStretchingStiffnessFixedVars(), opts, callback=cb, hessianShift = 0)
stretchingStiffness = inflation.getStretchingStiffness(ipu, betas, optimizer, 0, ipu.getStretchingStiffnessFixedVars())

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(False)

In [ ]:
%%capture
ss_obj = stretching_stiffness_class(ipu, ipu.sheet, optimizer, viewer, 2)
ss_obj.setVars(ss_obj.getVars())

In [ ]:
fd_validation.gradConvergencePlot(ss_obj)

In [ ]:
fd_validation.secondDerivativeConvergencePlot(ss_obj,  epsilons = np.logspace(-6, -2, 20))

In [ ]:
stiffness_optimizer = inflation.get_inflation_optimizer(az_ipu, [az_ipu.get_average_z_idx(), periodic_unit_helper.get_center_fixedVars(az_ipu.ipu)[0], periodic_unit_helper.get_center_fixedVars(az_ipu.ipu)[1], az_ipu.numVars() - 2, az_ipu.numVars() - 1], opts, callback=cb, hessianShift = 0)

In [ ]:
bs_obj = periodic_simulation_setup.bending_stiffness_class(az_ipu, az_ipu.ipu.sheet, az_optimizer, az_viewer, fixedVars = [])

bs_obj.setVars(bs_obj.getVars())

fd_validation.secondDerivativeConvergencePlot(bs_obj, epsilons = np.logspace(-6, -1, 50))